In [ ]:
import os
import glob
import json, time, shutil
import numpy as np

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

# =========================
# AUTO PATH RESOLUTION
# =========================
candidates = glob.glob('/kaggle/input/**/hard_negatives_30k.jsonl', recursive=True)

if not candidates:
    raise FileNotFoundError("hard_negatives_30k.jsonl bulunamadı. Dataset Kaggle input'a ekli değil veya isim farklı.")

HN_PATH = candidates[0]
print('[ok] HN_PATH:', HN_PATH)

BI_PATH    = '/kaggle/input/datasets/atakanakbaba/ft-code-5000-model'
CROSS_PATH = '/kaggle/working/codecrossenc-v2'
OUT_DIR    = '/kaggle/working/eval'

os.makedirs(CROSS_PATH, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader

# =========================
# TRAIN DATA
# =========================
print('[train] loading hard negatives...')

samples = []
with open(HN_PATH) as f:
    for line in f:
        row = json.loads(line)
        samples.append(InputExample(texts=[row['query'], row['pos']], label=1.0))
        for neg in [row.get('neg1'), row.get('neg2')]:
            if neg:
                samples.append(InputExample(texts=[row['query'], neg], label=0.0))

print('[train]', len(samples), 'training pairs')

cross = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-6-v2',
    num_labels=1,
    max_length=384
)

loader = DataLoader(samples, shuffle=True, batch_size=8)

t0 = time.time()
cross.fit(
    train_dataloader=loader,
    epochs=1,
    warmup_steps=100,
    output_path=CROSS_PATH,
    show_progress_bar=True
)

print('[train] done in', round(time.time()-t0, 1), 's')
print('[train] files:', os.listdir(CROSS_PATH))

from datasets import load_dataset

# =========================
# TEST DATA
# =========================
def load_test():
    ds = load_dataset('code_search_net', 'python', split='test')

    queries, truths, corpus_ids = [], [], []
    seen = set()

    for i, row in enumerate(ds):
        body = row.get('func_code_string') or ''
        doc = (row.get('func_documentation_string') or '').strip()

        if len(body) < 40 or len(doc) < 10:
            continue

        key = body[:200]
        if key in seen:
            continue
        seen.add(key)

        cid = 'ts' + str(i)
        queries.append(doc.splitlines()[0][:200])
        truths.append(cid)
        corpus_ids.append(cid)

    body_by_id = {
        'ts' + str(i): (row.get('func_code_string') or '')
        for i, row in enumerate(ds)
    }

    corpus_texts = [body_by_id[cid] for cid in corpus_ids]
    return queries, truths, corpus_ids, corpus_texts

t0 = time.time()
queries, truths, corpus_ids, corpus_texts = load_test()
print('[load]', len(queries), 'queries in', round(time.time()-t0, 1), 's')

from sentence_transformers import SentenceTransformer

TOP_K = 20

print('[bi] loading FT-Code-5000...')
bi = SentenceTransformer(BI_PATH)
bi.max_seq_length = 256

t0 = time.time()
corpus_emb = bi.encode(
    corpus_texts,
    batch_size=64,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

query_emb = bi.encode(
    queries,
    batch_size=64,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=True
)

print('[bi] encoded in', round(time.time()-t0, 1), 's')

n_q = len(queries)
cid_to_idx = {cid: i for i, cid in enumerate(corpus_ids)}
truth_idx_arr = np.array([cid_to_idx[t] for t in truths])

CHUNK_Q = 500
topk_idx = np.zeros((n_q, TOP_K), dtype=np.int32)

for qs in range(0, n_q, CHUNK_Q):
    qe = min(qs + CHUNK_Q, n_q)

    sims_chunk = query_emb[qs:qe] @ corpus_emb.T

    tu = np.argpartition(-sims_chunk, TOP_K, axis=1)[:, :TOP_K]
    rows = np.arange(qe - qs)[:, None]
    sc = sims_chunk[rows, tu]

    ord_ = np.argsort(-sc, axis=1)
    topk_idx[qs:qe] = tu[rows, ord_]

bi_r1 = bi_r5 = bi_r10 = 0
bi_mrr = 0.0

for qi in range(n_q):
    ranked = topk_idx[qi]
    pos_arr = np.where(ranked == truth_idx_arr[qi])[0]

    if len(pos_arr) == 0:
        continue

    pos = int(pos_arr[0])

    if pos == 0:  bi_r1  += 1
    if pos < 5:   bi_r5  += 1
    if pos < 10:  bi_r10 += 1
    bi_mrr += 1.0 / (pos + 1)

print('[bi-alone] R@1=' + str(round(bi_r1/n_q, 4)) +
      ' R@5=' + str(round(bi_r5/n_q, 4)) +
      ' R@10=' + str(round(bi_r10/n_q, 4)) +
      ' MRR=' + str(round(bi_mrr/n_q, 4)))

from sentence_transformers import CrossEncoder as CE

cfg_path = os.path.join(CROSS_PATH, 'config.json')
with open(cfg_path) as f:
    cfg = json.load(f)

if 'model_type' not in cfg:
    cfg['model_type'] = 'bert'
    with open(cfg_path, 'w') as f:
        json.dump(cfg, f)
    print('[fix] config.json patched')

print('[cross] loading CodeCrossEnc-v2...')
cross_eval = CE(CROSS_PATH, max_length=384)

pairs = [
    (queries[qi], corpus_texts[int(cidx)])
    for qi in range(n_q)
    for cidx in topk_idx[qi]
]

print('[cross]', len(pairs), 'pairs to score')

CHUNK = 2048
scores = np.zeros(len(pairs), dtype=np.float32)

for s in range(0, len(pairs), CHUNK):
    e = min(s + CHUNK, len(pairs))
    scores[s:e] = cross_eval.predict(pairs[s:e], batch_size=32)

scores = scores.reshape(n_q, TOP_K)
rerank_order = np.argsort(-scores, axis=1)
reranked_idx = topk_idx[np.arange(n_q)[:, None], rerank_order]

r1 = r5 = r10 = 0
mrr = 0.0

from pathlib import Path
out_path = Path(OUT_DIR) / 'per_query.jsonl'

with out_path.open('w') as f:
    for qi in range(n_q):
        ranked = reranked_idx[qi]
        pos_arr = np.where(ranked == truth_idx_arr[qi])[0]

        if len(pos_arr) == 0:
            rank = TOP_K + 1
        else:
            rank = int(pos_arr[0]) + 1
            if rank == 1:   r1  += 1
            if rank <= 5:   r5  += 1
            if rank <= 10:  r10 += 1
            mrr += 1.0 / rank

        f.write(json.dumps({
            'qi': qi,
            'truth': truths[qi],
            'rank': rank,
            'top1': corpus_ids[int(ranked[0])]
        }) + '\n')

summary = {
    'cross_checkpoint': 'codecrossenc-v2',
    'n_queries': n_q,
    'bi_alone':  {'R@1': bi_r1/n_q, 'R@5': bi_r5/n_q,
                  'R@10': bi_r10/n_q, 'MRR': bi_mrr/n_q},
    'reranked':  {'R@1': r1/n_q, 'R@5': r5/n_q,
                  'R@10': r10/n_q, 'MRR': mrr/n_q},
    'delta_R@1': (r1 - bi_r1) / n_q,
    'delta_MRR': (mrr - bi_mrr) / n_q,
}

(Path(OUT_DIR) / 'summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

bv = summary['bi_alone']
dv = summary['reranked']

print('\n=== KARAR ===')
print('bi-alone : R@1=' + str(round(bv['R@1'], 4)) + '  MRR=' + str(round(bv['MRR'], 4)))
print('v2 rerank: R@1=' + str(round(dv['R@1'], 4)) + '  MRR=' + str(round(dv['MRR'], 4)))
print('delta    : R@1=' + str(round(summary['delta_R@1'], 4)) + '  MRR=' + str(round(summary['delta_MRR'], 4)))

if dv['R@1'] >= 0.95:
    print('PASS (guclu): 3-axis probe (chunk x bi x cross) sirada.')
elif dv['R@1'] >= 0.93:
    print('PASS (marjinal): jphein followup postalabilir.')
else:
    print('FAIL: R@1=' + str(round(dv['R@1'], 4)) + ' < 0.93. Hard-neg budget artirilmali.')

In [ ]:
import subprocess; print(subprocess.check_output(['find', '/kaggle/input', '-type', 'f'], text=True))

In [ ]:
import os
  os.environ['CUDA_VISIBLE_DEVICES'] = '0'

  import json, time, shutil
  import numpy as np

  HN_PATH    = '/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/hard_negatives_30k.jsonl'
  BI_PATH    = '/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/ft-code-5000'
  CROSS_PATH = '/kaggle/working/codecrossenc-v2'
  OUT_DIR    = '/kaggle/working/eval'

  os.makedirs(CROSS_PATH, exist_ok=True)
  os.makedirs(OUT_DIR, exist_ok=True)

  from sentence_transformers import CrossEncoder, InputExample
  from torch.utils.data import DataLoader

  print('[train] loading hard negatives...')
  samples = []
  with open(HN_PATH) as f:
      for line in f:
          row = json.loads(line)
          samples.append(InputExample(texts=[row['query'], row['pos']], label=1.0))
          for neg in [row.get('neg1'), row.get('neg2')]:
              if neg:
                  samples.append(InputExample(texts=[row['query'], neg], label=0.0))
  print('[train]', len(samples), 'training pairs')

  cross = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', num_labels=1, max_length=384)
  loader = DataLoader(samples, shuffle=True, batch_size=8)
  t0 = time.time()
  cross.fit(train_dataloader=loader, epochs=1, warmup_steps=100,
            output_path=CROSS_PATH, show_progress_bar=True)
  print('[train] done in', round(time.time()-t0, 1), 's')
  print('[train] files:', os.listdir(CROSS_PATH))

  from datasets import load_dataset

  def load_test():
      ds = load_dataset('code_search_net', 'python', split='test')
      queries, truths, corpus_ids = [], [], []
      seen = set()
      for i, row in enumerate(ds):
          body = row.get('func_code_string') or ''
          doc = (row.get('func_documentation_string') or '').strip()
          if len(body) < 40 or len(doc) < 10:
              continue
          key = body[:200]
          if key in seen:
              continue
          seen.add(key)
          cid = 'ts' + str(i)
          queries.append(doc.splitlines()[0][:200])
          truths.append(cid)
          corpus_ids.append(cid)
      body_by_id = {'ts' + str(i): (row.get('func_code_string') or '') for i, row in enumerate(ds)}
      corpus_texts = [body_by_id[cid] for cid in corpus_ids]
      return queries, truths, corpus_ids, corpus_texts

  t0 = time.time()
  queries, truths, corpus_ids, corpus_texts = load_test()
  print('[load]', len(queries), 'queries in', round(time.time()-t0, 1), 's')

  from sentence_transformers import SentenceTransformer

  TOP_K = 20
  print('[bi] loading FT-Code-5000...')
  bi = SentenceTransformer(BI_PATH)
  bi.max_seq_length = 256

  t0 = time.time()
  corpus_emb = bi.encode(corpus_texts, batch_size=64, normalize_embeddings=True,
                         convert_to_numpy=True, show_progress_bar=True)
  query_emb  = bi.encode(queries, batch_size=64, normalize_embeddings=True,
                         convert_to_numpy=True, show_progress_bar=True)
  print('[bi] encoded in', round(time.time()-t0, 1), 's')

  n_q = len(queries)
  cid_to_idx = {cid: i for i, cid in enumerate(corpus_ids)}
  truth_idx_arr = np.array([cid_to_idx[t] for t in truths])

  CHUNK_Q = 500
  topk_idx = np.zeros((n_q, TOP_K), dtype=np.int32)
  for qs in range(0, n_q, CHUNK_Q):
      qe = min(qs + CHUNK_Q, n_q)
      sims_chunk = query_emb[qs:qe] @ corpus_emb.T
      tu = np.argpartition(-sims_chunk, TOP_K, axis=1)[:, :TOP_K]
      rows = np.arange(qe - qs)[:, None]
      sc = sims_chunk[rows, tu]
      ord_ = np.argsort(-sc, axis=1)
      topk_idx[qs:qe] = tu[rows, ord_]

  bi_r1 = bi_r5 = bi_r10 = 0
  bi_mrr = 0.0
  for qi in range(n_q):
      ranked = topk_idx[qi]
      pos_arr = np.where(ranked == truth_idx_arr[qi])[0]
      if len(pos_arr) == 0:
          continue
      pos = int(pos_arr[0])
      if pos == 0:  bi_r1  += 1
      if pos < 5:   bi_r5  += 1
      if pos < 10:  bi_r10 += 1
      bi_mrr += 1.0 / (pos + 1)
  print('[bi-alone] R@1=' + str(round(bi_r1/n_q, 4)) +
        ' R@5=' + str(round(bi_r5/n_q, 4)) +
        ' R@10=' + str(round(bi_r10/n_q, 4)) +
        ' MRR=' + str(round(bi_mrr/n_q, 4)))

  from sentence_transformers import CrossEncoder as CE

  cfg_path = os.path.join(CROSS_PATH, 'config.json')
  with open(cfg_path) as f:
      cfg = json.load(f)
  if 'model_type' not in cfg:
      cfg['model_type'] = 'bert'
      with open(cfg_path, 'w') as f:
          json.dump(cfg, f)
      print('[fix] config.json patched')

  print('[cross] loading CodeCrossEnc-v2...')
  cross_eval = CE(CROSS_PATH, max_length=384)

  t2 = time.time()
  pairs = [(queries[qi], corpus_texts[int(cidx)])
           for qi in range(n_q)
           for cidx in topk_idx[qi]]
  print('[cross]', len(pairs), 'pairs to score')

  CHUNK = 2048
  scores = np.zeros(len(pairs), dtype=np.float32)
  for s in range(0, len(pairs), CHUNK):
      e = min(s + CHUNK, len(pairs))
      scores[s:e] = cross_eval.predict(pairs[s:e], batch_size=32)

  scores = scores.reshape(n_q, TOP_K)
  rerank_order = np.argsort(-scores, axis=1)
  reranked_idx = topk_idx[np.arange(n_q)[:, None], rerank_order]

  r1 = r5 = r10 = 0
  mrr = 0.0
  from pathlib import Path
  out_path = Path(OUT_DIR) / 'per_query.jsonl'
  with out_path.open('w') as f:
      for qi in range(n_q):
          ranked = reranked_idx[qi]
          pos_arr = np.where(ranked == truth_idx_arr[qi])[0]
          if len(pos_arr) == 0:
              rank = TOP_K + 1
          else:
              rank = int(pos_arr[0]) + 1
              if rank == 1:   r1  += 1
              if rank <= 5:   r5  += 1
              if rank <= 10:  r10 += 1
              mrr += 1.0 / rank
          f.write(json.dumps({'qi': qi, 'truth': truths[qi], 'rank': rank,
                              'top1': corpus_ids[int(ranked[0])]}) + '\n')

  summary = {
      'cross_checkpoint': 'codecrossenc-v2',
      'n_queries': n_q,
      'bi_alone':  {'R@1': bi_r1/n_q, 'R@5': bi_r5/n_q,
                    'R@10': bi_r10/n_q, 'MRR': bi_mrr/n_q},
      'reranked':  {'R@1': r1/n_q, 'R@5': r5/n_q,
                    'R@10': r10/n_q, 'MRR': mrr/n_q},
      'delta_R@1': (r1 - bi_r1) / n_q,
      'delta_MRR': (mrr - bi_mrr) / n_q,
  }
  (Path(OUT_DIR) / 'summary.json').write_text(json.dumps(summary, indent=2))
  print(json.dumps(summary, indent=2))

  bv = summary['bi_alone']
  dv = summary['reranked']
  print('\n=== KARAR ===')
  print('bi-alone : R@1=' + str(round(bv['R@1'], 4)) + '  MRR=' + str(round(bv['MRR'], 4)))
  print('v2 rerank: R@1=' + str(round(dv['R@1'], 4)) + '  MRR=' + str(round(dv['MRR'], 4)))
  print('delta    : R@1=' + str(round(summary['delta_R@1'], 4)) + '  MRR=' + str(round(summary['delta_MRR'], 4)))
  if dv['R@1'] >= 0.95:
      print('PASS (guclu): 3-axis probe (chunk x bi x cross) sirada.')
  elif dv['R@1'] >= 0.93:
      print('PASS (marjinal): jphein followup postalabilir.')
  else:
      print('FAIL: R@1=' + str(round(dv['R@1'], 4)) + ' < 0.93. Hard-neg budget artirilmali.')

In [ ]:
%%writefile run.py
  import os
  os.environ['CUDA_VISIBLE_DEVICES'] = '0'

  import json, time, shutil
  import numpy as np

  HN_PATH    = '/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/hard_negatives_30k.jsonl'
  BI_PATH    = '/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/ft-code-5000'
  CROSS_PATH = '/kaggle/working/codecrossenc-v2'
  OUT_DIR    = '/kaggle/working/eval'

  os.makedirs(CROSS_PATH, exist_ok=True)
  os.makedirs(OUT_DIR, exist_ok=True)
  
  from sentence_transformers import CrossEncoder, InputExample
  from torch.utils.data import DataLoader

  print('[train] loading hard negatives...')
  samples = []
  with open(HN_PATH) as f:
      for line in f:
          row = json.loads(line)
          samples.append(InputExample(texts=[row['query'], row['pos']], label=1.0))
          for neg in [row.get('neg1'), row.get('neg2')]:
              if neg:
                  samples.append(InputExample(texts=[row['query'], neg], label=0.0))
  print('[train]', len(samples), 'training pairs')

  cross = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', num_labels=1, max_length=384)
  loader = DataLoader(samples, shuffle=True, batch_size=8)
  t0 = time.time()
  cross.fit(train_dataloader=loader, epochs=1, warmup_steps=100,
            output_path=CROSS_PATH, show_progress_bar=True)
  print('[train] done in', round(time.time()-t0, 1), 's')
  print('[train] files:', os.listdir(CROSS_PATH))

  from datasets import load_dataset

  def load_test():
      ds = load_dataset('code_search_net', 'python', split='test')
      queries, truths, corpus_ids = [], [], []
      seen = set()
      for i, row in enumerate(ds):
          body = row.get('func_code_string') or ''
          doc = (row.get('func_documentation_string') or '').strip()
          if len(body) < 40 or len(doc) < 10:
              continue
          key = body[:200]
          if key in seen:
              continue
          seen.add(key)
          cid = 'ts' + str(i)
          queries.append(doc.splitlines()[0][:200])
          truths.append(cid)
          corpus_ids.append(cid)
      body_by_id = {'ts' + str(i): (row.get('func_code_string') or '') for i, row in enumerate(ds)}
      corpus_texts = [body_by_id[cid] for cid in corpus_ids]
      return queries, truths, corpus_ids, corpus_texts

  t0 = time.time()
  queries, truths, corpus_ids, corpus_texts = load_test()
  print('[load]', len(queries), 'queries in', round(time.time()-t0, 1), 's')

  from sentence_transformers import SentenceTransformer

  TOP_K = 20
  print('[bi] loading FT-Code-5000...')
  bi = SentenceTransformer(BI_PATH)
  bi.max_seq_length = 256

  t0 = time.time()
  corpus_emb = bi.encode(corpus_texts, batch_size=64, normalize_embeddings=True,
                         convert_to_numpy=True, show_progress_bar=True)
  query_emb  = bi.encode(queries, batch_size=64, normalize_embeddings=True,
                         convert_to_numpy=True, show_progress_bar=True)
  print('[bi] encoded in', round(time.time()-t0, 1), 's')

  n_q = len(queries)
  cid_to_idx = {cid: i for i, cid in enumerate(corpus_ids)}
  truth_idx_arr = np.array([cid_to_idx[t] for t in truths])

  CHUNK_Q = 500
  topk_idx = np.zeros((n_q, TOP_K), dtype=np.int32)
  for qs in range(0, n_q, CHUNK_Q):
      qe = min(qs + CHUNK_Q, n_q)
      sims_chunk = query_emb[qs:qe] @ corpus_emb.T
      tu = np.argpartition(-sims_chunk, TOP_K, axis=1)[:, :TOP_K]
      rows = np.arange(qe - qs)[:, None]
      sc = sims_chunk[rows, tu]
      ord_ = np.argsort(-sc, axis=1)
      topk_idx[qs:qe] = tu[rows, ord_]

  bi_r1 = bi_r5 = bi_r10 = 0
  bi_mrr = 0.0
  for qi in range(n_q):
      ranked = topk_idx[qi]
      pos_arr = np.where(ranked == truth_idx_arr[qi])[0]
      if len(pos_arr) == 0:
          continue
      pos = int(pos_arr[0])
      if pos == 0:  bi_r1  += 1
      if pos < 5:   bi_r5  += 1
      if pos < 10:  bi_r10 += 1
      bi_mrr += 1.0 / (pos + 1)
  print('[bi-alone] R@1=' + str(round(bi_r1/n_q, 4)) +
        ' R@5=' + str(round(bi_r5/n_q, 4)) +
        ' R@10=' + str(round(bi_r10/n_q, 4)) +
        ' MRR=' + str(round(bi_mrr/n_q, 4)))

  from sentence_transformers import CrossEncoder as CE

  cfg_path = os.path.join(CROSS_PATH, 'config.json')
  with open(cfg_path) as f:
      cfg = json.load(f)
  if 'model_type' not in cfg:
      cfg['model_type'] = 'bert'
      with open(cfg_path, 'w') as f:
          json.dump(cfg, f)
      print('[fix] config.json patched')

  print('[cross] loading CodeCrossEnc-v2...')
  cross_eval = CE(CROSS_PATH, max_length=384)

  t2 = time.time()
  pairs = [(queries[qi], corpus_texts[int(cidx)])
           for qi in range(n_q)
           for cidx in topk_idx[qi]]
  print('[cross]', len(pairs), 'pairs to score')

  CHUNK = 2048
  scores = np.zeros(len(pairs), dtype=np.float32)
  for s in range(0, len(pairs), CHUNK):
      e = min(s + CHUNK, len(pairs))
      scores[s:e] = cross_eval.predict(pairs[s:e], batch_size=32)

  scores = scores.reshape(n_q, TOP_K)
  rerank_order = np.argsort(-scores, axis=1)
  reranked_idx = topk_idx[np.arange(n_q)[:, None], rerank_order]

  r1 = r5 = r10 = 0
  mrr = 0.0
  from pathlib import Path
  out_path = Path(OUT_DIR) / 'per_query.jsonl'
  with out_path.open('w') as f:
      for qi in range(n_q):
          ranked = reranked_idx[qi]
          pos_arr = np.where(ranked == truth_idx_arr[qi])[0]
          if len(pos_arr) == 0:
              rank = TOP_K + 1
          else:
              rank = int(pos_arr[0]) + 1
              if rank == 1:   r1  += 1
              if rank <= 5:   r5  += 1
              if rank <= 10:  r10 += 1
              mrr += 1.0 / rank
          f.write(json.dumps({'qi': qi, 'truth': truths[qi], 'rank': rank,
                              'top1': corpus_ids[int(ranked[0])]}) + '\n')

  summary = {
      'cross_checkpoint': 'codecrossenc-v2',
      'n_queries': n_q,
      'bi_alone':  {'R@1': bi_r1/n_q, 'R@5': bi_r5/n_q,
                    'R@10': bi_r10/n_q, 'MRR': bi_mrr/n_q},
      'reranked':  {'R@1': r1/n_q, 'R@5': r5/n_q,
                    'R@10': r10/n_q, 'MRR': mrr/n_q},
      'delta_R@1': (r1 - bi_r1) / n_q,
      'delta_MRR': (mrr - bi_mrr) / n_q,
  }
  (Path(OUT_DIR) / 'summary.json').write_text(json.dumps(summary, indent=2))
  print(json.dumps(summary, indent=2))

  bv = summary['bi_alone']
  dv = summary['reranked']
  print('\n=== KARAR ===')
  print('bi-alone : R@1=' + str(round(bv['R@1'], 4)) + '  MRR=' + str(round(bv['MRR'], 4)))
  print('v2 rerank: R@1=' + str(round(dv['R@1'], 4)) + '  MRR=' + str(round(dv['MRR'], 4)))
  print('delta    : R@1=' + str(round(summary['delta_R@1'], 4)) + '  MRR=' + str(round(summary['delta_MRR'], 4)))
  if dv['R@1'] >= 0.95:
      print('PASS (guclu): 3-axis probe (chunk x bi x cross) sirada.')
  elif dv['R@1'] >= 0.93:
      print('PASS (marjinal): jphein followup postalabilir.')
  else:
      print('FAIL: R@1=' + str(round(dv['R@1'], 4)) + ' < 0.93. Hard-neg budget artirilmali.')

In [ ]:
!python run.py

In [ ]:
%%writefile run.py
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import json, time, shutil
import numpy as np

HN_PATH    = '/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/hard_negatives_30k.jsonl'
BI_PATH    = '/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/ft-code-5000'
CROSS_PATH = '/kaggle/working/codecrossenc-v2'
OUT_DIR    = '/kaggle/working/eval'

os.makedirs(CROSS_PATH, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader

print('[train] loading hard negatives...')
samples = []
with open(HN_PATH) as f:
    for line in f:
        row = json.loads(line)
        samples.append(InputExample(texts=[row['query'], row['pos']], label=1.0))
        for neg in [row.get('neg1'), row.get('neg2')]:
            if neg:
                samples.append(InputExample(texts=[row['query'], neg], label=0.0))
print('[train]', len(samples), 'training pairs')

cross = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', num_labels=1, max_length=384)
loader = DataLoader(samples, shuffle=True, batch_size=8)
t0 = time.time()
cross.fit(train_dataloader=loader, epochs=1, warmup_steps=100,
          output_path=CROSS_PATH, show_progress_bar=True)
print('[train] done in', round(time.time()-t0, 1), 's')
print('[train] files:', os.listdir(CROSS_PATH))

from datasets import load_dataset

def load_test():
    ds = load_dataset('code_search_net', 'python', split='test')
    queries, truths, corpus_ids = [], [], []
    seen = set()
    for i, row in enumerate(ds):
        body = row.get('func_code_string') or ''
        doc = (row.get('func_documentation_string') or '').strip()
        if len(body) < 40 or len(doc) < 10:
            continue
        key = body[:200]
        if key in seen:
            continue
        seen.add(key)
        cid = 'ts' + str(i)
        queries.append(doc.splitlines()[0][:200])
        truths.append(cid)
        corpus_ids.append(cid)
    body_by_id = {'ts' + str(i): (row.get('func_code_string') or '') for i, row in enumerate(ds)}
    corpus_texts = [body_by_id[cid] for cid in corpus_ids]
    return queries, truths, corpus_ids, corpus_texts

t0 = time.time()
queries, truths, corpus_ids, corpus_texts = load_test()
print('[load]', len(queries), 'queries in', round(time.time()-t0, 1), 's')

from sentence_transformers import SentenceTransformer

TOP_K = 20
print('[bi] loading FT-Code-5000...')
bi = SentenceTransformer(BI_PATH)
bi.max_seq_length = 256

t0 = time.time()
corpus_emb = bi.encode(corpus_texts, batch_size=64, normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=True)
query_emb  = bi.encode(queries, batch_size=64, normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=True)
print('[bi] encoded in', round(time.time()-t0, 1), 's')

n_q = len(queries)
cid_to_idx = {cid: i for i, cid in enumerate(corpus_ids)}
truth_idx_arr = np.array([cid_to_idx[t] for t in truths])

CHUNK_Q = 500
topk_idx = np.zeros((n_q, TOP_K), dtype=np.int32)
for qs in range(0, n_q, CHUNK_Q):
    qe = min(qs + CHUNK_Q, n_q)
    sims_chunk = query_emb[qs:qe] @ corpus_emb.T
    tu = np.argpartition(-sims_chunk, TOP_K, axis=1)[:, :TOP_K]
    rows = np.arange(qe - qs)[:, None]
    sc = sims_chunk[rows, tu]
    ord_ = np.argsort(-sc, axis=1)
    topk_idx[qs:qe] = tu[rows, ord_]

bi_r1 = bi_r5 = bi_r10 = 0
bi_mrr = 0.0
for qi in range(n_q):
    ranked = topk_idx[qi]
    pos_arr = np.where(ranked == truth_idx_arr[qi])[0]
    if len(pos_arr) == 0:
        continue
    pos = int(pos_arr[0])
    if pos == 0:  bi_r1  += 1
    if pos < 5:   bi_r5  += 1
    if pos < 10:  bi_r10 += 1
    bi_mrr += 1.0 / (pos + 1)

print('[bi-alone] R@1=' + str(round(bi_r1/n_q, 4)) +
      ' R@5=' + str(round(bi_r5/n_q, 4)) +
      ' R@10=' + str(round(bi_r10/n_q, 4)) +
      ' MRR=' + str(round(bi_mrr/n_q, 4)))

from sentence_transformers import CrossEncoder as CE

cfg_path = os.path.join(CROSS_PATH, 'config.json')
with open(cfg_path) as f:
    cfg = json.load(f)
if 'model_type' not in cfg:
    cfg['model_type'] = 'bert'
    with open(cfg_path, 'w') as f:
        json.dump(cfg, f)
    print('[fix] config.json patched')

print('[cross] loading CodeCrossEnc-v2...')
cross_eval = CE(CROSS_PATH, max_length=384)

t2 = time.time()
pairs = [(queries[qi], corpus_texts[int(cidx)])
         for qi in range(n_q)
         for cidx in topk_idx[qi]]
print('[cross]', len(pairs), 'pairs to score')

CHUNK = 2048
scores = np.zeros(len(pairs), dtype=np.float32)
for s in range(0, len(pairs), CHUNK):
    e = min(s + CHUNK, len(pairs))
    scores[s:e] = cross_eval.predict(pairs[s:e], batch_size=32)

scores = scores.reshape(n_q, TOP_K)
rerank_order = np.argsort(-scores, axis=1)
reranked_idx = topk_idx[np.arange(n_q)[:, None], rerank_order]

r1 = r5 = r10 = 0
mrr = 0.0
from pathlib import Path
out_path = Path(OUT_DIR) / 'per_query.jsonl'
with out_path.open('w') as f:
    for qi in range(n_q):
        ranked = reranked_idx[qi]
        pos_arr = np.where(ranked == truth_idx_arr[qi])[0]
        if len(pos_arr) == 0:
            rank = TOP_K + 1
        else:
            rank = int(pos_arr[0]) + 1
            if rank == 1:   r1  += 1
            if rank <= 5:   r5  += 1
            if rank <= 10:  r10 += 1
            mrr += 1.0 / rank
        f.write(json.dumps({'qi': qi, 'truth': truths[qi], 'rank': rank,
                            'top1': corpus_ids[int(ranked[0])]}) + '\n')

summary = {
    'cross_checkpoint': 'codecrossenc-v2',
    'n_queries': n_q,
    'bi_alone':  {'R@1': bi_r1/n_q, 'R@5': bi_r5/n_q,
                  'R@10': bi_r10/n_q, 'MRR': bi_mrr/n_q},
    'reranked':  {'R@1': r1/n_q, 'R@5': r5/n_q,
                  'R@10': r10/n_q, 'MRR': mrr/n_q},
    'delta_R@1': (r1 - bi_r1) / n_q,
    'delta_MRR': (mrr - bi_mrr) / n_q,
}

(Path(OUT_DIR) / 'summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

bv = summary['bi_alone']
dv = summary['reranked']
print('\n=== KARAR ===')
print('bi-alone : R@1=' + str(round(bv['R@1'], 4)) + '  MRR=' + str(round(bv['MRR'], 4)))
print('v2 rerank: R@1=' + str(round(dv['R@1'], 4)) + '  MRR=' + str(round(dv['MRR'], 4)))
print('delta    : R@1=' + str(round(summary['delta_R@1'], 4)) + '  MRR=' + str(round(summary['delta_MRR'], 4)))
if dv['R@1'] >= 0.95:
    print('PASS (guclu): 3-axis probe (chunk x bi x cross) sirada.')
elif dv['R@1'] >= 0.93:
    print('PASS (marjinal): jphein followup postalabilir.')
else:
    print('FAIL: R@1=' + str(round(dv['R@1'], 4)) + ' < 0.93. Hard-neg budget artirilmali.')

In [2]:
 !pip show sentence-transformers

Name: sentence-transformers
Version: 5.5.0
Summary: Embeddings, Retrieval, and Reranking
Home-page: https://www.SBERT.net
Author: 
Author-email: Nils Reimers <info@nils-reimers.de>, Tom Aarsen <tom.aarsen@huggingface.co>
License: Apache 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, numpy, scikit-learn, scipy, torch, tqdm, transformers, typing_extensions
Required-by: 


In [4]:
%%writefile train.py
  import os, json, time
  os.environ['CUDA_VISIBLE_DEVICES'] = '0'
  
  HN_PATH    = '/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/hard_negatives_30k.jsonl'
  CROSS_PATH = '/kaggle/working/codecrossenc-v2'
  os.makedirs(CROSS_PATH, exist_ok=True)
  
  from sentence_transformers import CrossEncoder, InputExample
  from torch.utils.data import DataLoader
  
  print('[train] loading...')
  samples = []
  with open(HN_PATH) as f:
      for line in f: 
          row = json.loads(line)
          samples.append(InputExample(texts=[row['query'], row['pos']], label=1.0))
          for neg in [row.get('neg1'), row.get('neg2')]:
              if neg:
                  samples.append(InputExample(texts=[row['query'], neg], label=0.0))
  print('[train]', len(samples), 'pairs')
  
  cross = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', num_labels=1, max_length=384)
  loader = DataLoader(samples, shuffle=True, batch_size=8)
  t0 = time.time()
  cross.fit(train_dataloader=loader, epochs=1, warmup_steps=100,
            output_path=CROSS_PATH, show_progress_bar=True)
  cross.save(CROSS_PATH)
  print('[train] done in', round(time.time()-t0, 1), 's')
  print('[train] files:', os.listdir(CROSS_PATH))

Writing train.py


In [6]:
%%writefile eval.py
  import os, json, time
  import numpy as np
  from pathlib import Path

  BI_PATH    = '/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/ft-code-5000'
  CROSS_PATH = '/kaggle/working/codecrossenc-v2'
  OUT_DIR    = '/kaggle/working/eval'
  TOP_K      = 20
  os.makedirs(OUT_DIR, exist_ok=True)

  from datasets import load_dataset
  from sentence_transformers import SentenceTransformer

  def load_test():
      ds = load_dataset('code_search_net', 'python', split='test')
      queries, truths, corpus_ids = [], [], []
      seen = set()
      for i, row in enumerate(ds):
          body = row.get('func_code_string') or ''
          doc = (row.get('func_documentation_string') or '').strip()
          if len(body) < 40 or len(doc) < 10:
              continue
          key = body[:200]
          if key in seen:
              continue
          seen.add(key)
          cid = 'ts' + str(i)
          queries.append(doc.splitlines()[0][:200])
          truths.append(cid)
          corpus_ids.append(cid)
      body_by_id = {'ts' + str(i): (row.get('func_code_string') or '') for i, row in enumerate(ds)}
      corpus_texts = [body_by_id[cid] for cid in corpus_ids]
      return queries, truths, corpus_ids, corpus_texts

  queries, truths, corpus_ids, corpus_texts = load_test()
  print('[load]', len(queries), 'queries')
  
  bi = SentenceTransformer(BI_PATH)
  bi.max_seq_length = 256
  corpus_emb = bi.encode(corpus_texts, batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
  query_emb  = bi.encode(queries, batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)

  n_q = len(queries)
  cid_to_idx = {cid: i for i, cid in enumerate(corpus_ids)}
  truth_idx_arr = np.array([cid_to_idx[t] for t in truths])

  CHUNK_Q = 500
  topk_idx = np.zeros((n_q, TOP_K), dtype=np.int32)
  for qs in range(0, n_q, CHUNK_Q):
      qe = min(qs + CHUNK_Q, n_q)
      sims_chunk = query_emb[qs:qe] @ corpus_emb.T
      tu = np.argpartition(-sims_chunk, TOP_K, axis=1)[:, :TOP_K]
      rows = np.arange(qe - qs)[:, None]
      sc = sims_chunk[rows, tu]
      topk_idx[qs:qe] = tu[rows, np.argsort(-sc, axis=1)]

  bi_r1 = bi_r5 = bi_r10 = 0
  bi_mrr = 0.0
  for qi in range(n_q):
      ranked = topk_idx[qi]
      pos_arr = np.where(ranked == truth_idx_arr[qi])[0]
      if len(pos_arr) == 0:
          continue
      pos = int(pos_arr[0])
      if pos == 0:  bi_r1  += 1
      if pos < 5:   bi_r5  += 1
      if pos < 10:  bi_r10 += 1
      bi_mrr += 1.0 / (pos + 1)
  print('[bi-alone] R@1=' + str(round(bi_r1/n_q, 4)) + ' MRR=' + str(round(bi_mrr/n_q, 4)))

  from sentence_transformers import CrossEncoder as CE

  cfg_path = os.path.join(CROSS_PATH, 'config.json')
      cfg = json.load(f) 
  if 'model_type' not in cfg:
      cfg['model_type'] = 'bert'
      with open(cfg_path, 'w') as f:
          json.dump(cfg, f)
      print('[fix] config.json patched')
      
  cross_eval = CE(CROSS_PATH, max_length=384)
  pairs = [(queries[qi], corpus_texts[int(cidx)]) for qi in range(n_q) for cidx in topk_idx[qi]]
  print('[cross]', len(pairs), 'pairs')
  
  CHUNK = 2048
  scores = np.zeros(len(pairs), dtype=np.float32)
  for s in range(0, len(pairs), CHUNK):
      e = min(s + CHUNK, len(pairs))
      scores[s:e] = cross_eval.predict(pairs[s:e], batch_size=32)
      
  scores = scores.reshape(n_q, TOP_K)
  reranked_idx = topk_idx[np.arange(n_q)[:, None], np.argsort(-scores, axis=1)]
  
  r1 = r5 = r10 = 0
  mrr = 0.0 
  out_path = Path(OUT_DIR) / 'per_query.jsonl'
  with out_path.open('w') as f:
      for qi in range(n_q):
          ranked = reranked_idx[qi]
          pos_arr = np.where(ranked == truth_idx_arr[qi])[0]
          if len(pos_arr) == 0:
              rank = TOP_K + 1
          else:
              rank = int(pos_arr[0]) + 1
              if rank == 1:   r1  += 1 
              if rank <= 5:   r5  += 1
              if rank <= 10:  r10 += 1
              mrr += 1.0 / rank
          f.write(json.dumps({'qi': qi, 'truth': truths[qi], 'rank': rank, 'top1': corpus_ids[int(ranked[0])]}) + '\n')
          
  summary = {
      'n_queries': n_q,
      'bi_alone':  {'R@1': bi_r1/n_q, 'R@5': bi_r5/n_q, 'R@10': bi_r10/n_q, 'MRR': bi_mrr/n_q},
      'reranked':  {'R@1': r1/n_q,    'R@5': r5/n_q,    'R@10': r10/n_q,    'MRR': mrr/n_q},
      'delta_R@1': (r1 - bi_r1) / n_q,
      'delta_MRR': (mrr - bi_mrr) / n_q,
  }   
  (Path(OUT_DIR) / 'summary.json').write_text(json.dumps(summary, indent=2))
  print(json.dumps(summary, indent=2))
  
  bv = summary['bi_alone']
  dv = summary['reranked']
  print('\n=== KARAR ===')
  print('bi-alone : R@1=' + str(round(bv['R@1'], 4)) + '  MRR=' + str(round(bv['MRR'], 4)))
  print('reranked : R@1=' + str(round(dv['R@1'], 4)) + '  MRR=' + str(round(dv['MRR'], 4)))
  print('delta    : R@1=' + str(round(summary['delta_R@1'], 4)) + '  MRR=' + str(round(summary['delta_MRR'], 4)))
  if dv['R@1'] >= 0.95:
      print('PASS (guclu)')
  elif dv['R@1'] >= 0.93:
      print('PASS (marjinal)')
  else:
      print('FAIL: R@1=' + str(round(dv['R@1'], 4)) + ' < 0.93')

Overwriting eval.py


In [14]:
!head -5 /kaggle/working/train.py

!python train.py


In [18]:
 !cat /kaggle/working/train.py

!python train.py


In [20]:
import textwrap,pathlib
pathlib.Path('/kaggle/working/train.py').write_text(textwrap.dedent("""
  import os, json, time
  os.environ['CUDA_VISIBLE_DEVICES'] = '0'

  HN_PATH    = '/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/hard_negatives_30k.jsonl'
  CROSS_PATH = '/kaggle/working/codecrossenc-v2'
  os.makedirs(CROSS_PATH, exist_ok=True)

  from sentence_transformers import CrossEncoder, InputExample
  from torch.utils.data import DataLoader

  print('[train] loading...')
  samples = []
  with open(HN_PATH) as f:
      for line in f:
          row = json.loads(line)
          samples.append(InputExample(texts=[row['query'], row['pos']], label=1.0))
          for neg in [row.get('neg1'), row.get('neg2')]:
              if neg:
                  samples.append(InputExample(texts=[row['query'], neg], label=0.0))
  print('[train]', len(samples), 'pairs')

  cross = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', num_labels=1, max_length=384)
  loader = DataLoader(samples, shuffle=True, batch_size=8)
  t0 = time.time()
  cross.fit(train_dataloader=loader, epochs=1, warmup_steps=100,
            output_path=CROSS_PATH, show_progress_bar=True)
  cross.save(CROSS_PATH)
  print('[train] done in', round(time.time()-t0, 1), 's')
  print('[train] files:', os.listdir(CROSS_PATH))
""").lstrip())
print('yazildi')

yazildi


In [21]:
!python3 train.py

[train] loading...
[train] 90000 pairs
Loading weights: 100%|█| 105/105 [00:00<00:00, 1422.51it/s, Materializing param=
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
{'loss': '0.09776', 'grad_norm': '0.004639', 'learning_rate': '1.928e-05', 'epoch': '0.04444'}
{'loss': '0.08119', 'grad_norm': '0.03371', 'learning_rate': '1.839e-05', 'epoch': '0.08889'}
{'loss': '0.06984', 'grad_norm': '0.02096', 'learning_rate': '1.749e-05', 'epoch': '0.1333'}
{'loss': '0.0529', 'grad_norm': '0.09256', 'learning_rate': '1.659e-05', 'epoch': '0.1778'}
{'loss': '0.06032', 'grad_norm': '0.04967', 'learning_rate': '1.57e-05', 'epoch': '0.2222'}
{'loss': '0.05456', 'grad_norm': '0.002431', 'learning_rate': '1

In [5]:
import textwrap,pathlib
pathlib.Path('/kaggle/working/eval.py').write_text(textwrap.dedent("""
import os, json, time
import numpy as np
from pathlib import Path

BI_PATH    = '/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/ft-code-5000'
CROSS_PATH = '/kaggle/working/codecrossenc-v2'
OUT_DIR    = '/kaggle/working/eval'
TOP_K      = 20
os.makedirs(OUT_DIR, exist_ok=True)

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

def load_test():
  ds = load_dataset('code_search_net', 'python', split='test')
  queries, truths, corpus_ids = [], [], []
  seen = set()
  for i, row in enumerate(ds):
      body = row.get('func_code_string') or ''
      doc = (row.get('func_documentation_string') or '').strip()
      if len(body) < 40 or len(doc) < 10:
          continue
      key = body[:200]
      if key in seen:
          continue
      seen.add(key)
      cid = 'ts' + str(i)
      queries.append(doc.splitlines()[0][:200])
      truths.append(cid)
      corpus_ids.append(cid)
  body_by_id = {'ts' + str(i): (row.get('func_code_string') or '') for i, row in enumerate(ds)}
  corpus_texts = [body_by_id[cid] for cid in corpus_ids]
  return queries, truths, corpus_ids, corpus_texts

queries, truths, corpus_ids, corpus_texts = load_test()
print('[load]', len(queries), 'queries')

bi = SentenceTransformer(BI_PATH)
bi.max_seq_length = 256
corpus_emb = bi.encode(corpus_texts, batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
query_emb  = bi.encode(queries, batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)

n_q = len(queries)
cid_to_idx = {cid: i for i, cid in enumerate(corpus_ids)}
truth_idx_arr = np.array([cid_to_idx[t] for t in truths])

CHUNK_Q = 500
topk_idx = np.zeros((n_q, TOP_K), dtype=np.int32)
for qs in range(0, n_q, CHUNK_Q):
  qe = min(qs + CHUNK_Q, n_q)
  sims_chunk = query_emb[qs:qe] @ corpus_emb.T
  tu = np.argpartition(-sims_chunk, TOP_K, axis=1)[:, :TOP_K]
  rows = np.arange(qe - qs)[:, None]
  sc = sims_chunk[rows, tu]
  topk_idx[qs:qe] = tu[rows, np.argsort(-sc, axis=1)]

bi_r1 = bi_r5 = bi_r10 = 0
bi_mrr = 0.0
for qi in range(n_q):
  ranked = topk_idx[qi]
  pos_arr = np.where(ranked == truth_idx_arr[qi])[0]
  if len(pos_arr) == 0:
      continue
  pos = int(pos_arr[0])
  if pos == 0:  bi_r1  += 1
  if pos < 5:   bi_r5  += 1
  if pos < 10:  bi_r10 += 1
  bi_mrr += 1.0 / (pos + 1)
print('[bi-alone] R@1=' + str(round(bi_r1/n_q, 4)) + ' MRR=' + str(round(bi_mrr/n_q, 4)))

from sentence_transformers import CrossEncoder as CE
cross_eval = CE(CROSS_PATH, max_length=384)

pairs = [(queries[qi], corpus_texts[int(cidx)]) for qi in range(n_q) for cidx in topk_idx[qi]]
print('[cross]', len(pairs), 'pairs')

CHUNK = 2048
scores = np.zeros(len(pairs), dtype=np.float32)
for s in range(0, len(pairs), CHUNK):
  e = min(s + CHUNK, len(pairs))
  scores[s:e] = cross_eval.predict(pairs[s:e], batch_size=32)

scores = scores.reshape(n_q, TOP_K)
reranked_idx = topk_idx[np.arange(n_q)[:, None], np.argsort(-scores, axis=1)]

r1 = r5 = r10 = 0
mrr = 0.0
out_path = Path(OUT_DIR) / 'per_query.jsonl'
with out_path.open('w') as fout:
  for qi in range(n_q):
      ranked = reranked_idx[qi]
      pos_arr = np.where(ranked == truth_idx_arr[qi])[0]
      if len(pos_arr) == 0:
          rank = TOP_K + 1
      else:
          rank = int(pos_arr[0]) + 1
          if rank == 1:   r1  += 1
          if rank <= 5:   r5  += 1
          if rank <= 10:  r10 += 1
          mrr += 1.0 / rank
      fout.write(json.dumps({'qi': qi, 'truth': truths[qi], 'rank': rank, 'top1': corpus_ids[int(ranked[0])]}) + chr(10))

summary = {
  'n_queries': n_q,
  'bi_alone':  {'R@1': bi_r1/n_q, 'R@5': bi_r5/n_q, 'R@10': bi_r10/n_q, 'MRR': bi_mrr/n_q},
  'reranked':  {'R@1': r1/n_q,    'R@5': r5/n_q,    'R@10': r10/n_q,    'MRR': mrr/n_q},
  'delta_R@1': (r1 - bi_r1) / n_q,
  'delta_MRR': (mrr - bi_mrr) / n_q,
}
(Path(OUT_DIR) / 'summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

bv = summary['bi_alone']
dv = summary['reranked']
print('bi-alone : R@1=' + str(round(bv['R@1'], 4)) + '  MRR=' + str(round(bv['MRR'], 4)))
print('reranked : R@1=' + str(round(dv['R@1'], 4)) + '  MRR=' + str(round(dv['MRR'], 4)))
if dv['R@1'] >= 0.95:
  print('PASS (guclu)')
elif dv['R@1'] >= 0.93:
  print('PASS (marjinal)')
else:
  print('FAIL')
""").lstrip())
print('yazildi')

yazildi


In [11]:
import textwrap,pathlib
pathlib.Path('/kaggle/working/train.py').write_text(textwrap.dedent("""
import os, json, time
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

HN_PATH    = '/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/hard_negatives_30k.jsonl'
CROSS_PATH = '/kaggle/working/codecrossenc-v2'
os.makedirs(CROSS_PATH, exist_ok=True)

from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader

print('[train] loading...')
samples = []
with open(HN_PATH) as f:
  for line in f:
      row = json.loads(line)
      samples.append(InputExample(texts=[row['query'], row['pos']], label=1.0))
      for neg in [row.get('neg1'), row.get('neg2')]:
          if neg:
              samples.append(InputExample(texts=[row['query'], neg], label=0.0))
print('[train]', len(samples), 'pairs')

cross = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', num_labels=1, max_length=384)
loader = DataLoader(samples, shuffle=True, batch_size=8)
t0 = time.time()
cross.fit(train_dataloader=loader, epochs=1, warmup_steps=100,
        output_path=CROSS_PATH, show_progress_bar=True)
cross.save(CROSS_PATH)
print('[train] done in', round(time.time()-t0, 1), 's')
print('[train] files:', os.listdir(CROSS_PATH))
""").lstrip())
print('yazildi')

yazildi


In [7]:
!python3 eval.py

README.md: 14.1kB [00:00, 23.9MB/s]
python/train-00000-of-00001.parquet: 100%|███| 522M/522M [00:07<00:00, 66.1MB/s]
python/test-00000-of-00001.parquet: 100%|██| 28.7M/28.7M [00:00<00:00, 31.7MB/s]
python/validation-00000-of-00001.parquet: 100%|█| 30.7M/30.7M [00:00<00:00, 34.9
Generating train split: 100%|█| 412178/412178 [00:04<00:00, 102925.96 examples/s
Generating test split: 100%|███| 22176/22176 [00:00<00:00, 106542.75 examples/s]
Generating validation split: 100%|█| 23107/23107 [00:00<00:00, 105578.03 example
[load] 21935 queries
You are trying to use a model that was created with Sentence Transformers version 5.4.1, but you're currently using version 5.2.3. This might cause unexpected behavior or errors. In that case, try to update to the latest version.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sentence_transformers/util/misc.py", line 62, in import_from_string
    module = importlib.import_module(dotted_path)
             ^^^^^^^^^^^^^

In [8]:
import subprocess
subprocess.run(['pip', 'install', '-q', 'sentence-transformers>=5.4.1'], check=True)
print('kuruldu')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.7/588.7 kB 9.5 MB/s eta 0:00:00
kuruldu


In [13]:
!python3 train.py

[train] loading...
[train] 90000 pairs
config.json: 100%|█████████████████████████████| 794/794 [00:00<00:00, 4.22MB/s]
model.safetensors:   0%|                            | 0.00/90.9M [00:01<?, ?B/s]
Loading weights: 100%|█| 105/105 [00:00<00:00, 1315.83it/s, Materializing param=
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
tokenizer_config.json: 1.33kB [00:00, 4.46MB/s]
vocab.txt: 232kB [00:00, 14.4MB/s]
tokenizer.json: 711kB [00:00, 59.1MB/s]
special_tokens_map.json: 100%|██████████████████| 132/132 [00:00<00:00, 605kB/s]
{'loss': '0.1075', 'grad_norm': '31.01', 'learning_rate': '1.928e-05', 'epoch': '0.04444'}
{'loss': '0.06566', 'grad_norm': '0.005948', 'learning_rate': '1.839e-05'

In [14]:
import textwrap,pathlib
pathlib.Path('/kaggle/working/run_all.py').write_text(textwrap.dedent("""
import os, json, time
import numpy as np
from pathlib import Path

HN_PATH    = '/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/hard_negatives_30k.jsonl'
BI_PATH    = '/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/ft-code-5000'
CROSS_PATH = '/kaggle/working/codecrossenc-v2'
OUT_DIR    = '/kaggle/working/eval'
TOP_K      = 20

os.makedirs(CROSS_PATH, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

from sentence_transformers import CrossEncoder, InputExample, SentenceTransformer
from sentence_transformers import CrossEncoder as CE
from torch.utils.data import DataLoader

if not os.path.exists(os.path.join(CROSS_PATH, 'config.json')):
  print('[train] loading...')
  samples = []
  with open(HN_PATH) as f:
      for line in f:
          row = json.loads(line)
          samples.append(InputExample(texts=[row['query'], row['pos']], label=1.0))
          for neg in [row.get('neg1'), row.get('neg2')]:
              if neg:
                  samples.append(InputExample(texts=[row['query'], neg], label=0.0))
  print('[train]', len(samples), 'pairs')
  cross = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', num_labels=1, max_length=384)
  loader = DataLoader(samples, shuffle=True, batch_size=8)
  t0 = time.time()
  cross.fit(train_dataloader=loader, epochs=1, warmup_steps=100,
            output_path=CROSS_PATH, show_progress_bar=True)
  cross.save(CROSS_PATH)
  print('[train] done in', round(time.time()-t0, 1), 's')
else:
  print('[train] model zaten var, atlaniyor.')

print('[train] files:', os.listdir(CROSS_PATH))

from datasets import load_dataset

def load_test():
  ds = load_dataset('code_search_net', 'python', split='test')
  queries, truths, corpus_ids = [], [], []
  seen = set()
  for i, row in enumerate(ds):
      body = row.get('func_code_string') or ''
      doc = (row.get('func_documentation_string') or '').strip()
      if len(body) < 40 or len(doc) < 10:
          continue
      key = body[:200]
      if key in seen:
          continue
      seen.add(key)
      cid = 'ts' + str(i)
      queries.append(doc.splitlines()[0][:200])
      truths.append(cid)
      corpus_ids.append(cid)
  body_by_id = {'ts' + str(i): (row.get('func_code_string') or '') for i, row in enumerate(ds)}
  corpus_texts = [body_by_id[cid] for cid in corpus_ids]
  return queries, truths, corpus_ids, corpus_texts

queries, truths, corpus_ids, corpus_texts = load_test()
print('[load]', len(queries), 'queries')

bi = SentenceTransformer(BI_PATH)
bi.max_seq_length = 256
corpus_emb = bi.encode(corpus_texts, batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
query_emb  = bi.encode(queries, batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)

n_q = len(queries)
cid_to_idx = {cid: i for i, cid in enumerate(corpus_ids)}
truth_idx_arr = np.array([cid_to_idx[t] for t in truths])

CHUNK_Q = 500
topk_idx = np.zeros((n_q, TOP_K), dtype=np.int32)
for qs in range(0, n_q, CHUNK_Q):
  qe = min(qs + CHUNK_Q, n_q)
  sims_chunk = query_emb[qs:qe] @ corpus_emb.T
  tu = np.argpartition(-sims_chunk, TOP_K, axis=1)[:, :TOP_K]
  rows = np.arange(qe - qs)[:, None]
  sc = sims_chunk[rows, tu]
  topk_idx[qs:qe] = tu[rows, np.argsort(-sc, axis=1)]

bi_r1 = bi_r5 = bi_r10 = 0
bi_mrr = 0.0
for qi in range(n_q):
  ranked = topk_idx[qi]
  pos_arr = np.where(ranked == truth_idx_arr[qi])[0]
  if len(pos_arr) == 0:
      continue
  pos = int(pos_arr[0])
  if pos == 0:  bi_r1  += 1
  if pos < 5:   bi_r5  += 1
  if pos < 10:  bi_r10 += 1
  bi_mrr += 1.0 / (pos + 1)
print('[bi-alone] R@1=' + str(round(bi_r1/n_q, 4)) + ' MRR=' + str(round(bi_mrr/n_q, 4)))

cross_eval = CE(CROSS_PATH, max_length=384)
pairs = [(queries[qi], corpus_texts[int(cidx)]) for qi in range(n_q) for cidx in topk_idx[qi]]
print('[cross]', len(pairs), 'pairs')

CHUNK = 2048
scores = np.zeros(len(pairs), dtype=np.float32)
for s in range(0, len(pairs), CHUNK):
  e = min(s + CHUNK, len(pairs))
  scores[s:e] = cross_eval.predict(pairs[s:e], batch_size=32)

scores = scores.reshape(n_q, TOP_K)
reranked_idx = topk_idx[np.arange(n_q)[:, None], np.argsort(-scores, axis=1)]

r1 = r5 = r10 = 0
mrr = 0.0
out_path = Path(OUT_DIR) / 'per_query.jsonl'
with out_path.open('w') as fout:
  for qi in range(n_q):
      ranked = reranked_idx[qi]
      pos_arr = np.where(ranked == truth_idx_arr[qi])[0]
      if len(pos_arr) == 0:
          rank = TOP_K + 1
      else:
          rank = int(pos_arr[0]) + 1
          if rank == 1:   r1  += 1
          if rank <= 5:   r5  += 1
          if rank <= 10:  r10 += 1
          mrr += 1.0 / rank
      fout.write(json.dumps({'qi': qi, 'truth': truths[qi], 'rank': rank, 'top1': corpus_ids[int(ranked[0])]}) + chr(10))

summary = {
  'n_queries': n_q,
  'bi_alone':  {'R@1': bi_r1/n_q, 'R@5': bi_r5/n_q, 'R@10': bi_r10/n_q, 'MRR': bi_mrr/n_q},
  'reranked':  {'R@1': r1/n_q,    'R@5': r5/n_q,    'R@10': r10/n_q,    'MRR': mrr/n_q},
  'delta_R@1': (r1 - bi_r1) / n_q,
  'delta_MRR': (mrr - bi_mrr) / n_q,
}
(Path(OUT_DIR) / 'summary.json').write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

bv = summary['bi_alone']
dv = summary['reranked']
print('bi-alone : R@1=' + str(round(bv['R@1'], 4)) + '  MRR=' + str(round(bv['MRR'], 4)))
print('reranked : R@1=' + str(round(dv['R@1'], 4)) + '  MRR=' + str(round(dv['MRR'], 4)))
if dv['R@1'] >= 0.95:
  print('PASS (guclu)')
elif dv['R@1'] >= 0.93:
  print('PASS (marjinal)')
else:
  print('FAIL')
""").lstrip())
print('yazildi')

yazildi


In [17]:
import textwrap,pathlib
code = pathlib.Path('/kaggle/working/run_all.py').read_text()
code = code.replace('import os, json, time', 'import os, json, time' + chr(10) + "os.environ['CUDA_VISIBLE_DEVICES'] = '0'", 1)
pathlib.Path('/kaggle/working/run_all.py').write_text(code)
print('duzeltildi')

duzeltildi


In [18]:
!python3 run_all.py

[train] loading...
[train] 90000 pairs
Loading weights: 100%|█| 105/105 [00:00<00:00, 1506.21it/s, Materializing param=
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
{'loss': '0.1006', 'grad_norm': '0.0771', 'learning_rate': '1.928e-05', 'epoch': '0.04444'}
{'loss': '0.07732', 'grad_norm': '0.011', 'learning_rate': '1.839e-05', 'epoch': '0.08889'}
{'loss': '0.06405', 'grad_norm': '45.02', 'learning_rate': '1.749e-05', 'epoch': '0.1333'}
{'loss': '0.06282', 'grad_norm': '4.231', 'learning_rate': '1.659e-05', 'epoch': '0.1778'}
{'loss': '0.05674', 'grad_norm': '0.05263', 'learning_rate': '1.57e-05', 'epoch': '0.2222'}
{'loss': '0.05359', 'grad_norm': '0.09188', 'learning_rate': '1.48e-05',

In [24]:
import os
print(os.path.exists('/kaggle/working/eval/summary.json'))

True


In [ ]:
print(open('/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/hard_negatives_30k.jsonl').readline())

In [28]:
pathlib.Path('/kaggle/working/mine.py').write_text(textwrap.dedent("""
import os, json, time
import numpy as np
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

OUT_PATH = '/kaggle/working/hard_negatives_200k.jsonl'
BI_PATH  = '/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/ft-code-5000'
MAX_SAMPLES = 150000
TOP_K = 20
CHUNK_Q = 300

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

print('[mine] dataset yukleniyor...')
ds = load_dataset('code_search_net', 'python', split='train')

queries, codes = [], []
seen = set()
for row in ds:
if len(queries) >= MAX_SAMPLES:
break
body = row.get('func_code_string') or ''
doc = (row.get('func_documentation_string') or '').strip()
if len(body) < 40 or len(doc) < 10:
continue
key = body[:200]
if key in seen:
continue
seen.add(key)
queries.append(doc.splitlines()[0][:200])
codes.append(body)

print('[mine]', len(queries), 'pair filtrelendi')

bi = SentenceTransformer(BI_PATH)
bi.max_seq_length = 256

print('[mine] corpus encode ediliyor...')
t0 = time.time()
corpus_emb = bi.encode(codes, batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
query_emb  = bi.encode(queries, batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
print('[mine] encode bitti', round(time.time()-t0, 1), 's')

n_q = len(queries)
written = 0
t0 = time.time()

with open(OUT_PATH, 'w') as fout:
for qs in range(0, n_q, CHUNK_Q):
qe = min(qs + CHUNK_Q, n_q)
sims = query_emb[qs:qe] @ corpus_emb.T
for local_i in range(qe - qs):
global_i = qs + local_i
sim_row = sims[local_i].copy()
sim_row[global_i] = -1.0
top_idx = np.argpartition(-sim_row, TOP_K)[:TOP_K]
top_idx = top_idx[np.argsort(-sim_row[top_idx])]
negs = [codes[int(idx)] for idx in top_idx[:2]]
row_out = {
'query': queries[global_i],
'pos':   codes[global_i],
'neg1':  negs[0] if len(negs) > 0 else None,
'neg2':  negs[1] if len(negs) > 1 else None,
}
fout.write(json.dumps(row_out) + chr(10))
written += 1
if qs % 30000 == 0:
print('[mine]', qs, '/', n_q, '| yazilan:', written, '| sure:', round(time.time()-t0, 1), 's')

print('[mine] TAMAMLANDI:', written, 'pair ->', OUT_PATH)
""").lstrip())
print('yazildi')

yazildi


In [30]:
import textwrap,pathlib
pathlib.Path('/kaggle/working/mine.py').write_text(textwrap.dedent("""
import os, json, time
import numpy as np
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

OUT_PATH = '/kaggle/working/hard_negatives_200k.jsonl'
BI_PATH  = '/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/ft-code-5000'
MAX_SAMPLES = 150000
TOP_K = 20
CHUNK_Q = 300

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

print('[mine] dataset yukleniyor...')
ds = load_dataset('code_search_net', 'python', split='train')

queries, codes = [], []
seen = set()
for row in ds:
if len(queries) >= MAX_SAMPLES:
break
body = row.get('func_code_string') or ''
doc = (row.get('func_documentation_string') or '').strip()
if len(body) < 40 or len(doc) < 10:
continue
key = body[:200]
if key in seen:
continue
seen.add(key)
queries.append(doc.splitlines()[0][:200])
codes.append(body)

print('[mine]', len(queries), 'pair filtrelendi')

bi = SentenceTransformer(BI_PATH)
bi.max_seq_length = 256

print('[mine] corpus encode ediliyor...')
t0 = time.time()
corpus_emb = bi.encode(codes, batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
query_emb  = bi.encode(queries, batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
print('[mine] encode bitti', round(time.time()-t0, 1), 's')

n_q = len(queries)
written = 0
t0 = time.time()

with open(OUT_PATH, 'w') as fout:
for qs in range(0, n_q, CHUNK_Q):
qe = min(qs + CHUNK_Q, n_q)
sims = query_emb[qs:qe] @ corpus_emb.T
for local_i in range(qe - qs):
global_i = qs + local_i
sim_row = sims[local_i].copy()
sim_row[global_i] = -1.0
top_idx = np.argpartition(-sim_row, TOP_K)[:TOP_K]
top_idx = top_idx[np.argsort(-sim_row[top_idx])]
negs = [codes[int(idx)] for idx in top_idx[:2]]
row_out = {
'query': queries[global_i],
'pos':   codes[global_i],
'neg1':  negs[0] if len(negs) > 0 else None,
'neg2':  negs[1] if len(negs) > 1 else None,
}
fout.write(json.dumps(row_out) + chr(10))
written += 1
if qs % 30000 == 0:
print('[mine]', qs, '/', n_q, '| yazilan:', written, '| sure:', round(time.time()-t0, 1), 's')

print('[mine] TAMAMLANDI:', written, 'pair ->', OUT_PATH)
""").lstrip())
print('yazildi')

yazildi


In [26]:
print(open('/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/hard_negatives_30k.jsonl').readline())

{"query": "Estimate discontinuity in basis of low resolution image segmentation.", "pos": "def __msgc_step3_discontinuity_localization(self):\n        \"\"\"\n        Estimate discontinuity in basis of low resolution image segmentation.\n        :return: discontinuity in low resolution\n        \"\"\"\n        import scipy\n\n        start = self._start_time\n        seg = 1 - self.segmentation.astype(np.int8)\n        self.stats[\"low level object voxels\"] = np.sum(seg)\n        self.stats[\"low level image voxels\"] = np.prod(seg.shape)\n        # in seg is now stored low resolution segmentation\n        # back to normal parameters\n        # step 2: discontinuity localization\n        # self.segparams = sparams_hi\n        seg_border = scipy.ndimage.filters.laplace(seg, mode=\"constant\")\n        logger.debug(\"seg_border: %s\", scipy.stats.describe(seg_border, axis=None))\n        # logger.debug(str(np.max(seg_border)))\n        # logger.debug(str(np.min(seg_border)))\n        se

In [37]:
import textwrap,pathlib
  pathlib.Path('/kaggle/working/mine.py').write_text(textwrap.dedent("""
      import os, json, time
      import numpy as np
      os.environ['CUDA_VISIBLE_DEVICES'] = '0'

      OUT_PATH = '/kaggle/working/hard_negatives_200k.jsonl'
      BI_PATH  = '/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/ft-code-5000'
      MAX_SAMPLES = 150000
      TOP_K = 20
      CHUNK_Q = 300

      from datasets import load_dataset
      from sentence_transformers import SentenceTransformer

      print('[mine] dataset yukleniyor...')
      ds = load_dataset('code_search_net', 'python', split='train')

      queries, codes = [], []
      seen = set()
      for row in ds:
          if len(queries) >= MAX_SAMPLES:
              break
          body = row.get('func_code_string') or ''
          doc = (row.get('func_documentation_string') or '').strip()
          if len(body) < 40 or len(doc) < 10:
              continue
          key = body[:200]
          if key in seen:
              continue
          seen.add(key)
          queries.append(doc.splitlines()[0][:200])
          codes.append(body)

      print('[mine]', len(queries), 'pair filtrelendi')

      bi = SentenceTransformer(BI_PATH)
      bi.max_seq_length = 256

      print('[mine] corpus encode ediliyor...')
      t0 = time.time()
      corpus_emb = bi.encode(codes, batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
      query_emb  = bi.encode(queries, batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
      print('[mine] encode bitti', round(time.time()-t0, 1), 's')

      n_q = len(queries)
      written = 0
      t0 = time.time()

      with open(OUT_PATH, 'w') as fout:
          for qs in range(0, n_q, CHUNK_Q):
              qe = min(qs + CHUNK_Q, n_q)
              sims = query_emb[qs:qe] @ corpus_emb.T
              for local_i in range(qe - qs):
                  global_i = qs + local_i
                  sim_row = sims[local_i].copy()
                  sim_row[global_i] = -1.0
                  top_idx = np.argpartition(-sim_row, TOP_K)[:TOP_K]
                  top_idx = top_idx[np.argsort(-sim_row[top_idx])]
                  negs = [codes[int(idx)] for idx in top_idx[:2]]
                  row_out = {
                      'query': queries[global_i],
                      'pos':   codes[global_i],
                      'neg1':  negs[0] if len(negs) > 0 else None,
                      'neg2':  negs[1] if len(negs) > 1 else None,
                  }
                  fout.write(json.dumps(row_out) + chr(10))
                  written += 1
              if qs % 30000 == 0:
                  print('[mine]', qs, '/', n_q, '| yazilan:', written, '| sure:', round(time.time()-t0, 1), 's')

      print('[mine] TAMAMLANDI:', written, 'pair ->', OUT_PATH)
  """).lstrip())
  print('yazildi')

IndentationError: unexpected indent (3003927637.py, line 2)

In [31]:
import textwrap,pathlib
  pathlib.Path('/kaggle/working/mine.py').write_text(textwrap.dedent("""
      import os, json, time
      import numpy as np
      os.environ['CUDA_VISIBLE_DEVICES'] = '0'

      OUT_PATH = '/kaggle/working/hard_negatives_200k.jsonl'
      BI_PATH  = '/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/ft-code-5000'
      MAX_SAMPLES = 150000
      TOP_K = 20
      CHUNK_Q = 300

      from datasets import load_dataset
      from sentence_transformers import SentenceTransformer

      print('[mine] dataset yukleniyor...')
      ds = load_dataset('code_search_net', 'python', split='train')

      queries, codes = [], []
      seen = set()
      for row in ds:
          if len(queries) >= MAX_SAMPLES:
              break
          body = row.get('func_code_string') or ''
          doc = (row.get('func_documentation_string') or '').strip()
          if len(body) < 40 or len(doc) < 10:
              continue
          key = body[:200]
          if key in seen:
              continue
          seen.add(key)
          queries.append(doc.splitlines()[0][:200])
          codes.append(body)

      print('[mine]', len(queries), 'pair filtrelendi')

      bi = SentenceTransformer(BI_PATH)
      bi.max_seq_length = 256

      print('[mine] corpus encode ediliyor...')
      t0 = time.time()
      corpus_emb = bi.encode(codes, batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
      query_emb  = bi.encode(queries, batch_size=64, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=True)
      print('[mine] encode bitti', round(time.time()-t0, 1), 's')

      n_q = len(queries)
      written = 0
      t0 = time.time()

      with open(OUT_PATH, 'w') as fout:
          for qs in range(0, n_q, CHUNK_Q):
              qe = min(qs + CHUNK_Q, n_q)
              sims = query_emb[qs:qe] @ corpus_emb.T
              for local_i in range(qe - qs):
                  global_i = qs + local_i
                  sim_row = sims[local_i].copy()
                  sim_row[global_i] = -1.0
                  top_idx = np.argpartition(-sim_row, TOP_K)[:TOP_K]
                  top_idx = top_idx[np.argsort(-sim_row[top_idx])]
                  negs = [codes[int(idx)] for idx in top_idx[:2]]
                  row_out = {
                      'query': queries[global_i],
                      'pos':   codes[global_i],
                      'neg1':  negs[0] if len(negs) > 0 else None,
                      'neg2':  negs[1] if len(negs) > 1 else None,
                  }
                  fout.write(json.dumps(row_out) + chr(10))
                  written += 1
              if qs % 30000 == 0:
                  print('[mine]', qs, '/', n_q, '| yazilan:', written, '| sure:', round(time.time()-t0, 1), 's')

      print('[mine] TAMAMLANDI:', written, 'pair ->', OUT_PATH)
  """).lstrip())
  print('yazildi')

yazildi


In [39]:
!python3 mine.py

[mine] dataset yukleniyor...
[mine] 150000 pair filtrelendi
Loading weights: 100%|█| 103/103 [00:00<00:00, 1327.68it/s, Materializing param=
[mine] corpus encode ediliyor...
Batches: 100%|██████████████████████████████| 2344/2344 [00:42<00:00, 55.33it/s]
[mine] encode bitti 416.1 s
[mine] 0 / 150000 | yazilan: 300 | sure: 0.4 s
[mine] 30000 / 150000 | yazilan: 30300 | sure: 44.6 s
[mine] 60000 / 150000 | yazilan: 60300 | sure: 88.6 s
[mine] 90000 / 150000 | yazilan: 90300 | sure: 132.8 s
[mine] 120000 / 150000 | yazilan: 120300 | sure: 176.6 s
[mine] TAMAMLANDI: 150000 pair -> /kaggle/working/hard_negatives_200k.jsonl


In [42]:
import pathlib
code = pathlib.Path('/kaggle/working/run_all.py').read_text()
code = code.replace(
"'/kaggle/input/datasets/atakanakbaba/hard-negatives-30k-jsonl/hard_negatives_30k.jsonl'",
"'/kaggle/working/hard_negatives_200k.jsonl'"
).replace(
"if not os.path.exists(os.path.join(CROSS_PATH, 'config.json')):",
"if True:"
)
pathlib.Path('/kaggle/working/run_all.py').write_text(code)
print('guncellendi')

guncellendi


In [43]:
!python3 run_all.py

[train] loading...
[train] 450000 pairs
Loading weights: 100%|█| 105/105 [00:00<00:00, 1459.91it/s, Materializing param=
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
{'loss': '0.3593', 'grad_norm': '1.389', 'learning_rate': '1.986e-05', 'epoch': '0.008889'}
{'loss': '0.2594', 'grad_norm': '2.217', 'learning_rate': '1.968e-05', 'epoch': '0.01778'}
{'loss': '0.2459', 'grad_norm': '19.23', 'learning_rate': '1.95e-05', 'epoch': '0.02667'}
{'loss': '0.243', 'grad_norm': '3.815', 'learning_rate': '1.932e-05', 'epoch': '0.03556'}
{'loss': '0.2364', 'grad_norm': '27.37', 'learning_rate': '1.915e-05', 'epoch': '0.04444'}
{'loss': '0.2402', 'grad_norm': '0.7503', 'learning_rate': '1.897e-05', 'ep